In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [19]:
df = pd.read_csv("data/test.csv")

# Previous Position

In [20]:
df["prev_x_field"] = df.groupby("player_id")["x_field"].shift(1)
df["prev_y_field"] = df.groupby("player_id")["y_field"].shift(1)

# Calculate velocity components (v_x, v_y)
df["v_x"] = df["x_field"] - df["prev_x_field"]
df["v_y"] = df["y_field"] - df["prev_y_field"]

# Compute overall speed (magnitude of velocity vector)
df["speed"] = (df["v_x"]**2 + df["v_y"]**2) ** 0.5

In [21]:
# Determine movement direction based on velocity components
def get_direction(v_x, v_y):
    if pd.isna(v_x) or pd.isna(v_y):
        return None  # No movement detected
    
    angle = np.arctan2(v_y, v_x)  # Compute direction angle in radians
    angle_degrees = np.degrees(angle)  # Convert to degrees

    # Define movement categories based on angle
    if -22.5 <= angle_degrees < 22.5:
        return "Right"
    elif 22.5 <= angle_degrees < 67.5:
        return "Up-Right"
    elif 67.5 <= angle_degrees < 112.5:
        return "Up"
    elif 112.5 <= angle_degrees < 157.5:
        return "Up-Left"
    elif 157.5 <= angle_degrees or angle_degrees < -157.5:
        return "Left"
    elif -157.5 <= angle_degrees < -112.5:
        return "Down-Left"
    elif -112.5 <= angle_degrees < -67.5:
        return "Down"
    elif -67.5 <= angle_degrees < -22.5:
        return "Down-Right"
    else:
        return "Stationary"

# Apply function to determine direction
df["movement_direction"] = df.apply(lambda row: get_direction(row["v_x"], row["v_y"]), axis=1)
# Update movement direction to None if there is no movement (zero velocity)
df.fillna(0,inplace=True)
df.loc[(df["v_x"] == 0) & (df["v_y"] == 0), "movement_direction"] = None

In [22]:
# Update the function to determine the closest player using x_field and y_field for the ball
def find_closest_player_updated(row, players_df):
    if row["team"] == "Ball":
        ball_x, ball_y = row["x_field"], row["y_field"]
    else:
        return None  # Only determine possession for ball rows

    # Filter players from the dataset (excluding ball entries)
    player_positions = players_df[players_df["team"] != "Ball"].copy()

    if player_positions.empty:
        return None  # No players found

    # Compute distances from ball to each player
    player_positions["distance_to_ball"] = np.sqrt(
        (player_positions["x_field"] - ball_x) ** 2 +
        (player_positions["y_field"] - ball_y) ** 2
    )

    # Find the closest player
    closest_player = player_positions.loc[player_positions["distance_to_ball"].idxmin()]

    return closest_player["player_id"]

# Apply function only to rows where the object is the ball
df["player_in_possession"] = df.apply(lambda row: find_closest_player_updated(row, df), axis=1)

In [23]:
# Update ball action classification based on movement and closest player possession logic
def classify_ball_action_updated(row):
    if row["team"] != "Ball":
        return None  # Only classify actions for the ball

    if pd.isna(row["prev_x_field"]) or pd.isna(row["prev_y_field"]):
        return None  # No previous data to compare

    # Ball movement check
    if row["v_x"] == 0 and row["v_y"] == 0:
        return None  # Ball is stationary

    # Compute ball displacement
    ball_speed = row["speed"]

    # Check for possession change (based on closest player)
    if row["player_in_possession"] != row["attacker_with_ball"]:  # If attacker changes
        if ball_speed > 0 and ball_speed < 10:  # Threshold for pass distance
            return "Pass"
        elif ball_speed >= 10:  # Larger distance suggests a kick
            return "Kick"
        else:
            return "Possession Change (Static)"

    # If possession hasn't changed and the ball is moving, it's a carry
    if row["player_in_possession"] == row["attacker_with_ball"] and ball_speed > 0:
        return "Carry"

    return "Static"

# Apply classification function to the dataset
df["ball_action"] = df.apply(classify_ball_action_updated, axis=1)

In [24]:
# Refine Drop Goal detection to account for both left and right goal lines
def detect_scoring_event_final(row, prev_row):
    if row["team"] != "Ball":
        return None  # Only classify events for the ball

    # Assume field dimensions where x_field represents the horizontal field position
    try_line_left = 0    # Left team's try line
    try_line_right = 100  # Right team's try line (assuming 100m field length)

    # Goalpost y-range (adjust based on actual field data)
    goal_range_min = 30  # Minimum y-field position for a successful drop goal
    goal_range_max = 60  # Maximum y-field position for a successful drop goal

    # Try Detection: Ball crosses opponent's try line under control
    if row["player_in_possession"] is not None:
        if row["x_field"] <= try_line_left:  # R team scores on left
            return "Try"
        elif row["x_field"] >= try_line_right:  # L team scores on right
            return "Try"

    # Drop Goal Detection: Ensure ball was kicked and moving towards either goal line
    if row["ball_action"] == "Kick" and prev_row is not None:
        if (
            goal_range_min <= row["y_field"] <= goal_range_max  # Between goalposts
            and (
                (row["x_field"] < prev_row["x_field"] and row["x_field"] <= try_line_left)  # Moving left and crosses left try line
                or (row["x_field"] > prev_row["x_field"] and row["x_field"] >= try_line_right)  # Moving right and crosses right try line
            )
        ):
            return "Drop Goal"

    return None  # No scoring event detected

# Apply final scoring event detection using previous frame information
df["scoring_event"] = df.apply(lambda row: detect_scoring_event_final(row, df.shift(1).iloc[row.name]), axis=1)

In [25]:
# Define function to detect turnovers based on possession_id
def detect_turnover_by_possession(row, prev_row):
    if row["team"] != "Ball":
        return None  # Only classify events for the ball

    # Turnover occurs if the possession_id changes from the previous frame
    if prev_row is not None and row["possession_id"] is not None:
        if prev_row["possession_id"] is not None and row["possession_id"] != prev_row["possession_id"]:
            return "Turnover"

    return None  # No turnover detected

# Apply turnover detection using possession_id
df["turnover_event"] = df.apply(lambda row: detect_turnover_by_possession(row, df.shift(1).iloc[row.name]), axis=1)

In [26]:
# Define reward function with team-based movement penalties
def calculate_rewards_team_based(row):
    reward = 0

    # Reward for scoring
    if row["scoring_event"] == "Try":
        reward += 10
    elif row["scoring_event"] == "Drop Goal":
        reward += 5

    # Reward for passing (encouraging team play)
    if row["ball_action"] == "Pass":
        reward += 1

    # Team-based movement rewards and penalties
    if row["v_x"] is not None:
        if row["team"] == "R":  # R team attacks left (negative x)
            if row["v_x"] < 0:  # Moving left (forward)
                reward += abs(row["v_x"]) * 0.5
            elif row["v_x"] > 0:  # Moving right (backward)
                reward -= abs(row["v_x"]) * 0.25

        elif row["team"] == "L":  # L team attacks right (positive x)
            if row["v_x"] > 0:  # Moving right (forward)
                reward += abs(row["v_x"]) * 0.5
            elif row["v_x"] < 0:  # Moving left (backward)
                reward -= abs(row["v_x"]) * 0.25

    # Penalty for turnovers
    if row["turnover_event"] == "Turnover":
        reward -= 5

    return reward

# Apply team-based reward calculation
df["reward"] = df.apply(calculate_rewards_team_based, axis=1)

In [ ]:
import torch

# Function to extract full state representation for a single ball frame
def extract_state(row, players_df):
    """
    Build a state tensor for a ball row using only players from that same frame.
    Dimensions: 5 (ball) + N_players*6 (per player) + 5 (context)
    """
    state = []

    # Ball features (5)
    state.extend([row["x_field"], row["y_field"], row["v_x"], row["v_y"]])
    state.append(row["possession_id"] if pd.notna(row["possession_id"]) else -1)

    # Player features (6 per player) — only players from this frame
    for _, player in players_df.iterrows():
        state.extend([
            player["x_field"], player["y_field"],
            player["v_x"], player["v_y"],
            1 if player["team"] == "R" else -1,           # Team encoding
            1 if player["player_id"] == row["player_in_possession"] else 0  # In possession
        ])

    # Game context (5)
    state.append(1 if row["turnover_event"] == "Turnover" else 0)
    state.append(1 if row["ball_action"] == "Pass" else 0)
    state.append(1 if row["ball_action"] == "Carry" else 0)
    state.append(1 if row["ball_action"] == "Kick" else 0)
    state.append(
        1 if row["scoring_event"] == "Try"
        else (2 if row["scoring_event"] == "Drop Goal" else 0)
    )

    return torch.tensor(state, dtype=torch.float32)

# Extract state tensors using only each frame's own players (not all players globally)
df["state"] = df.apply(
    lambda row: extract_state(
        row,
        df[(df["frame_num"] == row["frame_num"]) & (df["team"] != "Ball")]
    ),
    axis=1
)

print(f"State tensors computed. Sample size: {df['state'].iloc[0].shape[0]} dimensions")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class Actor(nn.Module):
    def __init__(self, state_dim, hidden_dim=128):
        super(Actor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 3),   # Pass / Carry / Kick
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        return self.net(state)

# ── Step 1: Compute state_dim dynamically from real data ─────────────────────
sample_ball = df[df["team"] == "Ball"].iloc[0]
sample_players = df[
    (df["frame_num"] == sample_ball["frame_num"]) & (df["team"] != "Ball")
]
state_dim = extract_state(sample_ball, sample_players).shape[0]
print(f"Computed state_dim: {state_dim}  "
      f"(5 ball + {len(sample_players)}×6 player + 5 context)")

# ── Step 2: Build dataset from real game frames ───────────────────────────────
action_map = {"Pass": 0, "Carry": 1, "Kick": 2}
ball_rows = df[
    (df["team"] == "Ball") & (df["ball_action"].isin(action_map.keys()))
]

states, labels = [], []
for _, row in ball_rows.iterrows():
    frame_players = df[
        (df["frame_num"] == row["frame_num"]) & (df["team"] != "Ball")
    ]
    s = extract_state(row, frame_players)
    # Pad or truncate to canonical state_dim (handles frames with different player counts)
    if s.shape[0] < state_dim:
        s = torch.cat([s, torch.zeros(state_dim - s.shape[0])])
    else:
        s = s[:state_dim]
    states.append(s)
    labels.append(action_map[row["ball_action"]])

if len(states) == 0:
    print("\nNo labeled ball actions (Pass/Carry/Kick) found in this dataset.")
    print("This is expected with very few frames — run the full pipeline on a longer video.")
else:
    states_t = torch.stack(states)    # shape: (N, state_dim)
    labels_t = torch.tensor(labels)   # shape: (N,)
    print(f"Training samples: {len(states_t)}")

    # ── Step 3: Train Actor on real game states ───────────────────────────────
    model = Actor(state_dim)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    num_epochs = 200
    for epoch in range(num_epochs):
        model.train()
        probs = model(states_t)
        loss = loss_fn(probs, labels_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d}/{num_epochs} | Loss: {loss.item():.4f}")

    print("\nTraining complete on real game data!")


In [ ]:
# Run inference using the trained model
# Uses state_dim computed dynamically above
if 'model' in dir() and model is not None:
    model.eval()
    with torch.no_grad():
        sample_state = torch.randn(1, state_dim)  # Replace with a real game state
        action_probs = model(sample_state)
        chosen_action = torch.argmax(action_probs, dim=-1).item()
        action_names = ['Pass', 'Carry', 'Kick']
        print(f"Predicted Ball Action: {action_names[chosen_action]}")
        print(f"Probabilities: {action_probs}")
else:
    print("Model not trained yet — run the training cell above first.")
